# AudioTools Function Testing Notebook

This notebook demonstrates and tests all major audiotools functions using real audio files from the tests directory.

## Setup and Imports

In [ ]:
import audiotools
from audiotools import AudioSignal
from audiotools.data import transforms as tfm
from audiotools.core import util
from audiotools import metrics
import torch
import torchaudio
torchaudio.set_audio_backend("sox_io")
import numpy as np
import matplotlib.pyplot as plt
import os

# Set random seed for reproducibility
util.seed(42)

print(f"AudioTools version: {audiotools.__version__}")
print(f"PyTorch version: {torch.__version__}")

## 1. Loading Audio Files

Test different ways to load audio from the test directory.

In [ ]:
# Load speech audio
speech = AudioSignal("tests/audio/spk/f10_script4_produced.wav")
print("=" * 60)
print("SPEECH SIGNAL")
print("=" * 60)
print(f"  File path: tests/audio/spk/f10_script4_produced.wav")
print(f"  Sample rate: {speech.sample_rate} Hz")
print(f"  Duration: {speech.duration:.4f} seconds")
print(f"  Signal length: {speech.signal_length} samples")
print(f"  Audio shape: {speech.audio_data.shape} (batch, channels, samples)")
print(f"  Batch size: {speech.batch_size}")
print(f"  Num channels: {speech.num_channels}")
print(f"  Device: {speech.device}")
print(f"  Data type: {speech.audio_data.dtype}")
print()

# Load noise audio
noise = AudioSignal("tests/audio/nz/f5_script2_ipad_balcony1_room_tone.wav")
print("=" * 60)
print("NOISE SIGNAL")
print("=" * 60)
print(f"  File path: tests/audio/nz/f5_script2_ipad_balcony1_room_tone.wav")
print(f"  Sample rate: {noise.sample_rate} Hz")
print(f"  Duration: {noise.duration:.4f} seconds")
print(f"  Signal length: {noise.signal_length} samples")
print(f"  Audio shape: {noise.audio_data.shape} (batch, channels, samples)")
print(f"  Batch size: {noise.batch_size}")
print(f"  Num channels: {noise.num_channels}")
print()

# Load impulse response
ir = AudioSignal("tests/audio/ir/h179_Bar_1txts.wav")
print("=" * 60)
print("IMPULSE RESPONSE SIGNAL")
print("=" * 60)
print(f"  File path: tests/audio/ir/h179_Bar_1txts.wav")
print(f"  Sample rate: {ir.sample_rate} Hz")
print(f"  Duration: {ir.duration:.4f} seconds")
print(f"  Signal length: {ir.signal_length} samples")
print(f"  Audio shape: {ir.audio_data.shape} (batch, channels, samples)")
print(f"  Batch size: {ir.batch_size}")
print(f"  Num channels: {ir.num_channels}")

In [ ]:
# Load with offset and duration
print("\n" + "=" * 60)
print("LOADING WITH OFFSET AND DURATION")
print("=" * 60)
print("Input parameters:")
print(f"  File: tests/audio/spk/f10_script4_produced.wav")
print(f"  Offset: 2 seconds")
print(f"  Duration: 10 seconds")
print()

speech_excerpt = AudioSignal("tests/audio/spk/f10_script4_produced.wav", offset=2, duration=10)
print("Output AudioSignal:")
print(f"  Duration: {speech_excerpt.duration:.4f} seconds")
print(f"  Sample rate: {speech_excerpt.sample_rate} Hz")
print(f"  Signal length: {speech_excerpt.signal_length} samples")
print(f"  Audio shape: {speech_excerpt.audio_data.shape}")
print(f"  Expected samples: {10 * speech_excerpt.sample_rate} (duration * sr)")

In [ ]:
# Load from CSV sources
speech_sources = util.read_sources(["tests/audio/spk.csv"])
noise_sources = util.read_sources(["tests/audio/noises.csv"])
ir_sources = util.read_sources(["tests/audio/irs.csv"])

print(f"Found {len(speech_sources)} speech files")
print(f"Found {len(noise_sources)} noise files")
print(f"Found {len(ir_sources)} IR files")

### Additional Audio Properties

AudioSignal objects have many useful properties beyond basic shape and duration.

In [ ]:
print("=" * 60)
print("COMPREHENSIVE AUDIO PROPERTIES")
print("=" * 60)
print(f"Signal: speech_excerpt")
print()

print("Shape Information:")
print(f"  audio_data.shape: {speech_excerpt.audio_data.shape} (batch, channels, samples)")
print(f"  batch_size: {speech_excerpt.batch_size}")
print(f"  num_channels: {speech_excerpt.num_channels}")
print(f"  signal_length: {speech_excerpt.signal_length} samples")
print()

print("Time Information:")
print(f"  sample_rate: {speech_excerpt.sample_rate} Hz")
print(f"  duration: {speech_excerpt.duration:.6f} seconds")
print(f"  signal_duration: {speech_excerpt.signal_duration:.6f} seconds")
print()

print("Data Information:")
print(f"  device: {speech_excerpt.device}")
print(f"  dtype: {speech_excerpt.audio_data.dtype}")
print(f"  min value: {speech_excerpt.audio_data.min().item():.6f}")
print(f"  max value: {speech_excerpt.audio_data.max().item():.6f}")
print(f"  mean value: {speech_excerpt.audio_data.mean().item():.6f}")
print(f"  std value: {speech_excerpt.audio_data.std().item():.6f}")
print()

print("Loudness:")
loudness_val = speech_excerpt.loudness()
print(f"  ITU-R BS.1770 loudness: {loudness_val.item():.2f} LUFS")
print()

# Check if signal has STFT computed
speech_test = speech_excerpt.clone()
print("STFT State:")
print(f"  Has STFT (before .stft()): {speech_test.stft_data is not None}")
speech_test.stft()
print(f"  Has STFT (after .stft()): {speech_test.stft_data is not None}")
print(f"  STFT shape: {speech_test.stft_data.shape}")

## 2. Visualization

Test waveform and spectrogram plotting functions.

### Advanced Spectrogram Display

Test different specshow parameters for advanced visualization.

In [ ]:
print("=" * 60)
print("ADVANCED SPECTROGRAM DISPLAY OPTIONS")
print("=" * 60)

# Test different y_axis options
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Regular linear frequency
speech_excerpt.specshow(ax=axes[0], y_axis="linear")
axes[0].set_title("Linear Frequency Scale")

# Mel frequency scale
speech_excerpt.specshow(ax=axes[1], y_axis="mel")
axes[1].set_title("Mel Frequency Scale")

# With preemphasis (emphasizes high frequencies)
speech_excerpt.specshow(ax=axes[2], preemphasis=True)
axes[2].set_title("With Preemphasis")

plt.tight_layout()
plt.show()

print("specshow() supports different y_axis scales: 'linear', 'log', 'mel'")

### Save Images

Save spectrograms and waveforms as images.

In [ ]:
print("=" * 60)
print("SAVE SPECTROGRAM AS IMAGE")
print("=" * 60)

# Save spectrogram as image file
output_path = "/tmp/spectrogram.png"
speech_excerpt.save_image(output_path)

print(f"Saved spectrogram image to: {output_path}")
print(f"File exists: {os.path.exists(output_path)}")

# Check file size
if os.path.exists(output_path):
    file_size = os.path.getsize(output_path)
    print(f"File size: {file_size / 1024:.2f} KB")
print()
print("save_image() creates a visualization combining waveform and spectrogram")

In [ ]:
# Waveform plot
fig, ax = plt.subplots(figsize=(12, 3))
speech_excerpt.waveplot(ax=ax)
plt.title("Speech Waveform")
plt.tight_layout()
plt.show()

In [ ]:
# Spectrogram plot
fig, ax = plt.subplots(figsize=(12, 4))
speech_excerpt.specshow(ax=ax)
plt.title("Speech Spectrogram")
plt.tight_layout()
plt.show()

In [ ]:
# Combined waveform and spectrogram
speech_excerpt.wavespec()
plt.suptitle("Speech Waveform and Spectrogram")
plt.tight_layout()
plt.show()

## 3. Basic Signal Operations

Test cloning, copying, resampling, and mono conversion.

In [ ]:
# Clone signal
print("=" * 60)
print("CLONE OPERATION")
print("=" * 60)
print(f"Original shape: {speech_excerpt.audio_data.shape}")
speech_clone = speech_excerpt.clone()
print(f"Cloned shape: {speech_clone.audio_data.shape}")
print(f"Same shape: {speech_clone.audio_data.shape == speech_excerpt.audio_data.shape}")
print(f"Different object: {speech_clone is not speech_excerpt}")
print()

# Resample to different sample rate
print("=" * 60)
print("RESAMPLE OPERATION")
print("=" * 60)
print(f"Input:")
print(f"  Sample rate: {speech_excerpt.sample_rate} Hz")
print(f"  Shape: {speech_excerpt.audio_data.shape}")
print(f"  Signal length: {speech_excerpt.signal_length} samples")
print()

speech_resampled = speech_excerpt.clone().resample(16000)
print(f"Output (resampled to 16kHz):")
print(f"  Sample rate: {speech_resampled.sample_rate} Hz")
print(f"  Shape: {speech_resampled.audio_data.shape}")
print(f"  Signal length: {speech_resampled.signal_length} samples")
print(f"  Duration: {speech_resampled.duration:.4f}s (should stay the same)")
print()

# Convert to mono
print("=" * 60)
print("TO_MONO OPERATION")
print("=" * 60)
print(f"Input:")
print(f"  Shape: {speech_excerpt.audio_data.shape}")
print(f"  Num channels: {speech_excerpt.num_channels}")
print()

speech_mono = speech_excerpt.clone().to_mono()
print(f"Output:")
print(f"  Shape: {speech_mono.audio_data.shape}")
print(f"  Num channels: {speech_mono.num_channels}")

In [ ]:
# Extract excerpt
random_excerpt = speech.excerpt(audio_path="tests/audio/spk/f10_script4_produced.wav", duration=10.0)
print(f"Random excerpt duration: {random_excerpt.duration:.2f}s")

# Extract salient (loudest) excerpt
salient_excerpt = speech.salient_excerpt(audio_path="tests/audio/spk/f10_script4_produced.wav", duration=10.0)
print(f"Salient excerpt duration: {salient_excerpt.duration:.2f}s")

In [ ]:
# Padding and trimming
padded = speech_excerpt.clone().zero_pad(100, 100)
print(f"Padded signal length: {padded.signal_length} samples")

trimmed = padded.trim(100, 100)
print(f"Trimmed signal length: {trimmed.signal_length} samples")

## 4. DSP Operations

Test filtering, windowing, and other DSP functions.

In [ ]:
# Low-pass filter
speech_lowpass = speech_excerpt.clone().low_pass(4000)
print(f"Low-pass filtered at 4kHz")

# High-pass filter
speech_highpass = speech_excerpt.clone().high_pass(200)
print(f"High-pass filtered at 200Hz")

# Visualize filtered signals
fig, axes = plt.subplots(3, 1, figsize=(16, 8))
speech_excerpt.specshow(ax=axes[0])
axes[0].set_title("Original")
speech_lowpass.specshow(ax=axes[1])
axes[1].set_title("Low-pass (4kHz)")
speech_highpass.specshow(ax=axes[2])
axes[2].set_title("High-pass (200Hz)")
plt.tight_layout()
plt.show()

In [ ]:
# Volume change
print("=" * 60)
print("VOLUME CHANGE")
print("=" * 60)
print(f"Input:")
print(f"  Shape: {speech_excerpt.audio_data.shape}")
print(f"  Max amplitude: {speech_excerpt.audio_data.abs().max().item():.6f}")
print()

speech_louder = speech_excerpt.clone().volume_change(db=6)
speech_quieter = speech_excerpt.clone().volume_change(db=-6)

print(f"After +6dB volume change:")
print(f"  Shape: {speech_louder.audio_data.shape}")
print(f"  Max amplitude: {speech_louder.audio_data.abs().max().item():.6f}")
print(f"  Amplitude ratio: {(speech_louder.audio_data.abs().max() / speech_excerpt.audio_data.abs().max()).item():.4f}")
print()

print(f"After -6dB volume change:")
print(f"  Shape: {speech_quieter.audio_data.shape}")
print(f"  Max amplitude: {speech_quieter.audio_data.abs().max().item():.6f}")
print(f"  Amplitude ratio: {(speech_quieter.audio_data.abs().max() / speech_excerpt.audio_data.abs().max()).item():.4f}")
print()

# Normalize to target loudness
print("-" * 60)
print("NORMALIZE TO TARGET LOUDNESS")
print("-" * 60)
original_loudness = speech_excerpt.loudness()
print(f"Original loudness: {original_loudness.item():.2f} LUFS")

speech_normalized = speech_excerpt.clone().normalize(db=-20)
normalized_loudness = speech_normalized.loudness()

print(f"After normalize(db=-20):")
print(f"  Target loudness: -20 LUFS")
print(f"  Actual loudness: {normalized_loudness.item():.2f} LUFS")
print(f"  Shape: {speech_normalized.audio_data.shape}")

In [ ]:
# Pitch shift
print("=" * 60)
print("PITCH SHIFT")
print("=" * 60)
print(f"Input:")
print(f"  Duration: {speech_excerpt.duration:.4f}s")
print(f"  Shape: {speech_excerpt.audio_data.shape}")
print()

speech_pitched_up = speech_excerpt.clone().pitch_shift(n_semitones=4)
speech_pitched_down = speech_excerpt.clone().pitch_shift(n_semitones=-4)

print(f"Pitch shift +4 semitones (major third up):")
print(f"  Duration: {speech_pitched_up.duration:.4f}s")
print(f"  Shape: {speech_pitched_up.audio_data.shape}")
print(f"  Duration preserved: {abs(speech_pitched_up.duration - speech_excerpt.duration) < 0.01}")
print()

print(f"Pitch shift -4 semitones (major third down):")
print(f"  Duration: {speech_pitched_down.duration:.4f}s")
print(f"  Shape: {speech_pitched_down.audio_data.shape}")
print(f"  Duration preserved: {abs(speech_pitched_down.duration - speech_excerpt.duration) < 0.01}")

In [ ]:
# Time stretch
print("=" * 60)
print("TIME STRETCH")
print("=" * 60)
print(f"Input:")
print(f"  Duration: {speech_excerpt.duration:.4f}s")
print(f"  Shape: {speech_excerpt.audio_data.shape}")
print(f"  Signal length: {speech_excerpt.signal_length} samples")
print()

speech_faster = speech_excerpt.clone().time_stretch(factor=1.2)
speech_slower = speech_excerpt.clone().time_stretch(factor=0.8)

print(f"Time stretch 1.2x (20% faster):")
print(f"  Duration: {speech_faster.duration:.4f}s")
print(f"  Shape: {speech_faster.audio_data.shape}")
print(f"  Signal length: {speech_faster.signal_length} samples")
print(f"  Expected duration: {speech_excerpt.duration / 1.2:.4f}s")
print()

print(f"Time stretch 0.8x (20% slower):")
print(f"  Duration: {speech_slower.duration:.4f}s")
print(f"  Shape: {speech_slower.audio_data.shape}")
print(f"  Signal length: {speech_slower.signal_length} samples")
print(f"  Expected duration: {speech_excerpt.duration / 0.8:.4f}s")
print()

print("Note: Pitch is preserved, only duration changes")

In [ ]:
# Pitch shift
speech_pitched_up = speech_excerpt.clone().pitch_shift(n_semitones=4)
speech_pitched_down = speech_excerpt.clone().pitch_shift(n_semitones=-4)
print(f"Pitch shifted by +4 and -4 semitones")

In [ ]:
# Time stretch
speech_faster = speech_excerpt.clone().time_stretch(factor=1.2)
speech_slower = speech_excerpt.clone().time_stretch(factor=0.8)
print(f"Original duration: {speech_excerpt.duration:.2f}s")
print(f"Faster (1.2x): {speech_faster.duration:.2f}s")
print(f"Slower (0.8x): {speech_slower.duration:.2f}s")

In [ ]:
# Mix signals with SNR control
noise_excerpt = noise.excerpt(audio_path="tests/audio/nz/f5_script2_ipad_balcony1_room_tone.wav", duration=speech_excerpt.duration)
speech_with_noise = speech_excerpt.clone().mix(noise_excerpt, snr=10)
print(f"Mixed speech with noise at SNR=10dB")

# Visualize mixed signal
fig, axes = plt.subplots(3, 1, figsize=(12, 8))
speech_excerpt.specshow(ax=axes[0])
axes[0].set_title("Clean Speech")
noise_excerpt.specshow(ax=axes[1])
axes[1].set_title("Noise")
speech_with_noise.specshow(ax=axes[2])
axes[2].set_title("Speech + Noise (SNR=10dB)")
plt.tight_layout()
plt.show()

In [ ]:
# Windowing
windows = speech_excerpt.clone().windows(window_duration=0.1, hop_duration=0.05)
for i, window in enumerate(windows):
    print(f"Window {i}: shape {window.shape}")
    break  # Print only the first window for brevity

# Collect windows
window_list = speech_excerpt.clone().collect_windows(window_duration=0.1, hop_duration=0.05)
print(f"windows: {window_list.shape}")
print(f"First window shape: {window_list[0].audio_data.shape}")

In [ ]:
# Apply impulse response (convolution)
speech_reverb = speech_excerpt.clone().apply_ir(ir, drr=0)
print(f"Applied impulse response (reverb)")

# Visualize reverb effect
fig, axes = plt.subplots(2, 1, figsize=(12, 6))
speech_excerpt.specshow(ax=axes[0])
axes[0].set_title("Dry Speech")
speech_reverb.specshow(ax=axes[1])
axes[1].set_title("Speech with Reverb")
plt.tight_layout()
plt.show()

In [ ]:
# Codec simulation
speech_codec = speech_excerpt.clone().apply_codec("MP3", bits_per_sample="64k")
print(f"Applied MP3 codec at 64kbps")

In [ ]:
# Distortion effects
speech_clipped = speech_excerpt.clone().clip_distortion(clip_percentile=0.2)
print(f"Applied clipping distortion")
print(f"speech_clipped: {speech_clipped.audio_data}")
speech_quantized = speech_excerpt.clone().quantization(quantization_channels=8)
print(f"speech_quantized: {speech_quantized.audio_data}")
print(f"Applied 8-bit quantization")

speech_mulaw = speech_excerpt.clone().mulaw_quantization(quantization_channels=8)
print(f"speech_mulaw: {speech_mulaw.audio_data}")
print(f"Applied mu-law quantization")

## 6. STFT and Spectral Operations

Test STFT, mel spectrograms, and spectral manipulations.

In [ ]:
# STFT
print("=" * 60)
print("STFT (Short-Time Fourier Transform)")
print("=" * 60)
print(f"Input:")
print(f"  Audio shape: {speech_excerpt.audio_data.shape}")
print(f"  Sample rate: {speech_excerpt.sample_rate} Hz")
print(f"  Duration: {speech_excerpt.duration:.4f}s")
print()

speech_stft = speech_excerpt.clone()
speech_stft.stft()

print(f"After STFT:")
print(f"  STFT data shape: {speech_stft.stft_data.shape} (batch, ch, freq_bins, time_frames)")
print(f"  Magnitude shape: {speech_stft.magnitude.shape}")
print(f"  Phase shape: {speech_stft.phase.shape}")
print(f"  Frequency bins: {speech_stft.stft_data.shape[2]}")
print(f"  Time frames: {speech_stft.stft_data.shape[3]}")
print()

# Inverse STFT
speech_stft.istft()
print(f"After iSTFT (reconstruction):")
print(f"  Reconstructed audio shape: {speech_stft.audio_data.shape}")
print(f"  Original audio shape: {speech_excerpt.audio_data.shape}")
print(f"  Shape match: {speech_stft.audio_data.shape == speech_excerpt.audio_data.shape}")

In [ ]:
# Mel spectrogram
print("=" * 60)
print("MEL SPECTROGRAM & MFCC")
print("=" * 60)
print(f"Input audio shape: {speech_excerpt.audio_data.shape}")
print()

n_mels = 80
mel_spec = speech_excerpt.mel_spectrogram(n_mels=n_mels)
print(f"Mel Spectrogram:")
print(f"  n_mels: {n_mels}")
print(f"  Output shape: {mel_spec.shape} (batch, channels, mel_bins, time_frames)")
print(f"  Mel bins: {mel_spec.shape[2]}")
print(f"  Time frames: {mel_spec.shape[3]}")
print()

# MFCC
n_mfcc = 13
mfcc = speech_excerpt.mfcc(n_mfcc=n_mfcc)
print(f"MFCC (Mel-Frequency Cepstral Coefficients):")
print(f"  n_mfcc: {n_mfcc}")
print(f"  Output shape: {mfcc.shape} (batch, channels, n_mfcc, time_frames)")
print(f"  MFCC coefficients: {mfcc.shape[2]}")
print(f"  Time frames: {mfcc.shape[3]}")
print()

# Visualize mel spectrogram
fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(mel_spec[0][0].cpu().numpy(), aspect='auto', origin='lower', cmap='viridis')
plt.colorbar(im, ax=ax)
ax.set_title("Mel Spectrogram")
ax.set_xlabel("Time")
ax.set_ylabel("Mel Bin")
plt.tight_layout()
plt.show()

In [ ]:
# Phase manipulation
speech_phase_shifted = speech_excerpt.clone().shift_phase(shift=np.pi/4)
print(f"Phase shifted by π/4")

# speech_phase_inverted = speech_excerpt.clone()
# print(f"Phase inverted")

speech_phase_corrupt = speech_excerpt.clone().corrupt_phase(scale=0.5)
print(f"Phase corrupted")

In [ ]:
# Masking
speech_freq_masked = speech_excerpt.clone().mask_frequencies(fmin_hz=1000, fmax_hz=3000).istft()
print(f"Masked frequencies 1-3kHz")

speech_time_masked = speech_excerpt.clone().mask_timesteps(tmin_s=5.5, tmax_s=6.5).istft()
print(f"Masked time 5.5-6.5s")

speech_mag_masked = speech_excerpt.clone().mask_low_magnitudes(db_cutoff=-40).istft()
print(f"Masked low magnitudes below -40dB")

# Visualize masking
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
speech_excerpt.specshow(ax=axes[0, 0])
axes[0, 0].set_title("Original")
speech_freq_masked.specshow(ax=axes[0, 1])
axes[0, 1].set_title("Frequency Masked (1-3kHz)")
speech_time_masked.specshow(ax=axes[1, 0])
axes[1, 0].set_title("Time Masked (5.5-6.5s)")
speech_mag_masked.specshow(ax=axes[1, 1])
axes[1, 1].set_title("Magnitude Masked (<-40dB)")
plt.tight_layout()
plt.show()

## 7. Loudness Operations

Test loudness measurement and normalization.

In [ ]:
# Measure loudness
loudness = speech_excerpt.loudness()
print(f"Loudness: {loudness} LUFS")

# Normalize to target loudness
speech_norm_20 = speech_excerpt.clone().normalize(db=-20)
loudness_norm = speech_norm_20.loudness()
print(f"Normalized loudness: {loudness_norm} LUFS")

# Ensure max of audio
speech_safe = speech_excerpt.clone().ensure_max_of_audio()
print(f"Max value after ensuring: {speech_safe.audio_data.abs().max():.4f}")

## 8. Transforms

Test data augmentation transforms.

In [ ]:
# Room impulse response transform
rir_transform = tfm.RoomImpulseResponse(sources=["tests/audio/irs.csv"])
kwargs = rir_transform.instantiate(state=42, signal=speech_excerpt)
speech_rir = rir_transform(speech_excerpt.clone(), **kwargs)
print(f"Applied RIR transform")

# Background noise transform
bg_noise_transform = tfm.BackgroundNoise(sources=["tests/audio/noises.csv"], snr=("uniform", 5, 20))
kwargs = bg_noise_transform.instantiate(state=42, signal=speech_excerpt)
speech_bg_noise = bg_noise_transform(speech_excerpt.clone(), **kwargs)
print(f"Applied background noise transform")

In [ ]:
# Compose multiple transforms
composed_transform = tfm.Compose([
    tfm.VolumeChange(db=("uniform", -6, 6)),
    tfm.RoomImpulseResponse(sources=["tests/audio/irs.csv"]),
    tfm.BackgroundNoise(sources=["tests/audio/noises.csv"], snr=("uniform", 5, 15)),
    tfm.LowPass(cutoff=("choice", [4000, 8000, 16000])),
])

kwargs = composed_transform.instantiate(state=42, signal=speech_excerpt)
speech_augmented = composed_transform(speech_excerpt.clone(), **kwargs)
print(f"Applied composed transform")

# Visualize augmentation pipeline
fig, axes = plt.subplots(2, 1, figsize=(12, 6))
speech_excerpt.specshow(ax=axes[0])
axes[0].set_title("Original")
speech_augmented.specshow(ax=axes[1])
axes[1].set_title("Augmented (Volume + Reverb + Noise + LowPass)")
plt.tight_layout()
plt.show()

In [ ]:
# Test various single transforms
transforms_to_test = [
    ("Clipping Distortion", tfm.ClippingDistortion(perc=("uniform", 0.1, 0.3))),
    ("Quantization", tfm.Quantization(("choice", [8, 32, 128, 256, 1024]))),
    ("Equalizer", tfm.Equalizer(eq_amount=("const", 10.0))),
    ("Low Pass", tfm.LowPass(cutoff=("choice", [4000, 8000, 16000]))),
    ("High Pass", tfm.HighPass(cutoff=("choice", [50, 100, 250, 500, 1000]))),
    ("Frequency Mask", tfm.FrequencyMask(f_center=("uniform", 0, 1), f_width=("const", 0.1), prob = 1)),
    ("Time Mask", tfm.TimeMask(t_center=("uniform", 0, 0.1), t_width = ("const", 0.025), prob = 1)),
]

fig, axes = plt.subplots(4, 2, figsize=(14, 12))
axes = axes.flatten()

speech_excerpt.specshow(ax=axes[0])
axes[0].set_title("Original")

for i, (name, transform) in enumerate(transforms_to_test, start=1):
    kwargs = transform.instantiate(state=42, signal=speech_excerpt)
    transformed = transform(speech_excerpt.clone(), **kwargs)
    transformed.specshow(ax=axes[i])
    axes[i].set_title(name)
    
plt.tight_layout()
plt.show()

### Choose Transform

The Choose transform randomly selects one transform from a list to apply (unlike Compose which applies all).

In [ ]:
# Choose transform - picks ONE transform from the list
print("=" * 60)
print("CHOOSE TRANSFORM - Pick one: HighPass OR LowPass")
print("=" * 60)

choose_transform = tfm.Choose([
    tfm.HighPass(cutoff=("const", 1000)),
    tfm.LowPass(cutoff=("const", 4000)),
])

# Test with different seeds to show randomness
for seed in [0, 1, 2]:
    print(f"\nSeed {seed}:")
    kwargs = choose_transform.instantiate(state=seed, signal=speech_excerpt)
    print(f"  Instantiated kwargs keys: {list(kwargs.keys())}")
    output = choose_transform(speech_excerpt.clone(), **kwargs)
    print(f"  Input shape: {speech_excerpt.audio_data.shape}")
    print(f"  Output shape: {output.audio_data.shape}")
    print(f"  Applied transform: {[k for k in kwargs.keys() if 'Choose' not in k]}")

### Transform Filtering

You can name transforms and use filter() to selectively apply only certain named transforms.

In [ ]:
# Create named transform groups
print("=" * 60)
print("TRANSFORM FILTERING WITH NAMED GROUPS")
print("=" * 60)

group_a = tfm.Compose([
    tfm.LowPass(cutoff=("const", 4000)),
    tfm.VolumeChange(db=("const", -6)),
], name="group_a")

group_b = tfm.Compose([
    tfm.HighPass(cutoff=("const", 500)),
    tfm.Quantization(("const", 32)),
], name="group_b")

full_transform = tfm.Compose([group_a, group_b])

# Instantiate parameters
kwargs = full_transform.instantiate(state=42, signal=speech_excerpt)

print("\nFull kwargs structure:")
for key in kwargs.keys():
    print(f"  {key}")

# Apply all transforms
print("\n" + "-" * 60)
print("Applying ALL transforms:")
output_all = full_transform(speech_excerpt.clone(), **kwargs)
print(f"  Input shape: {speech_excerpt.audio_data.shape}")
print(f"  Output shape: {output_all.audio_data.shape}")

# Apply only group_a
print("\n" + "-" * 60)
print("Applying ONLY group_a (LowPass + VolumeChange):")
with full_transform.filter("group_a"):
    output_a = full_transform(speech_excerpt.clone(), **kwargs)
print(f"  Input shape: {speech_excerpt.audio_data.shape}")
print(f"  Output shape: {output_a.audio_data.shape}")

# Apply only group_b
print("\n" + "-" * 60)
print("Applying ONLY group_b (HighPass + Quantization):")
with full_transform.filter("group_b"):
    output_b = full_transform(speech_excerpt.clone(), **kwargs)
print(f"  Input shape: {speech_excerpt.audio_data.shape}")
print(f"  Output shape: {output_b.audio_data.shape}")

## 9b. Spectral Losses

AudioTools provides several spectral-based loss functions for training neural networks.

In [ ]:
# Import spectral metrics
from audiotools.metrics import spectral

print("=" * 60)
print("SPECTRAL LOSS FUNCTIONS")
print("=" * 60)

# Create two signals for comparison
signal_ref = speech_excerpt.clone()
signal_est = speech_excerpt.clone().low_pass(4000)  # Degraded version

print(f"Reference signal shape: {signal_ref.audio_data.shape}")
print(f"Estimate signal shape: {signal_est.audio_data.shape}")
print()

# 1. MultiScaleSTFTLoss
print("-" * 60)
print("MultiScaleSTFTLoss")
print("-" * 60)
ms_stft_loss = spectral.MultiScaleSTFTLoss(window_lengths=[2048, 512])
loss_val = ms_stft_loss(signal_est, signal_ref)
print(f"  Window lengths: [2048, 512]")
print(f"  Loss value: {loss_val.item():.6f}")
print(f"  Description: Computes STFT loss at multiple scales")
print()

# Identity check (same signal)
loss_identity = ms_stft_loss(signal_ref, signal_ref)
print(f"  Identity loss (same signal): {loss_identity.item():.6f}")
print()

# 2. MelSpectrogramLoss  
print("-" * 60)
print("MelSpectrogramLoss")
print("-" * 60)
mel_loss = spectral.MelSpectrogramLoss(n_mels=[150, 80], window_lengths=[2048, 512])
loss_val = mel_loss(signal_est, signal_ref)
print(f"  n_mels: [150, 80]")
print(f"  Window lengths: [2048, 512]")
print(f"  Loss value: {loss_val.item():.6f}")
print(f"  Description: Computes mel spectrogram distance")
print()

# 3. PhaseLoss
print("-" * 60)
print("PhaseLoss")
print("-" * 60)
phase_loss = spectral.PhaseLoss(window_length=2048, hop_length=512)
loss_val = phase_loss(signal_est, signal_ref)
print(f"  Window length: 2048")
print(f"  Hop length: 512")
print(f"  Loss value: {loss_val.item():.6f}")
print(f"  Description: Weighted phase difference loss")
print()

print("These loss functions are useful for training neural audio models!")

### Batch Instantiate

Use batch_instantiate() to create parameters for an entire batch at once.

In [ ]:
# Create a batch of signals
print("=" * 60)
print("BATCH INSTANTIATE")
print("=" * 60)

batch_size = 4
signal_list = [speech.excerpt(audio_path="tests/audio/spk/f10_script4_produced.wav", duration=2.0) for _ in range(batch_size)]
signal_batch = AudioSignal.batch(signal_list)

print(f"Created batch:")
print(f"  Batch size: {signal_batch.batch_size}")
print(f"  Batch shape: {signal_batch.audio_data.shape}")
print()

# Create a transform with randomness
transform = tfm.Compose([
    tfm.VolumeChange(db=("uniform", -10, 10)),
    tfm.LowPass(cutoff=("choice", [4000, 8000, 16000])),
])

# Use batch_instantiate with multiple seeds
seeds = range(batch_size)
batch_kwargs = transform.batch_instantiate(seeds, signal=signal_batch)

print("Batch instantiated parameters:")
for key, value in batch_kwargs.items():
    if isinstance(value, dict):
        print(f"  {key}:")
        for subkey, subvalue in value.items():
            if isinstance(subvalue, torch.Tensor):
                print(f"    {subkey}: shape={subvalue.shape}, values={subvalue.tolist()}")
    elif isinstance(value, torch.Tensor):
        print(f"  {key}: shape={value.shape}")
print()

# Apply transform to entire batch at once
output_batch = transform(signal_batch.clone(), **batch_kwargs)

print(f"Batch transform applied:")
print(f"  Input batch shape: {signal_batch.audio_data.shape}")
print(f"  Output batch shape: {output_batch.audio_data.shape}")
print(f"  Each item got different random parameters based on its seed")

## 9. Audio Tables and Display Functions

AudioTools provides functions for creating interactive audio tables in notebooks.

In [ ]:
from audiotools import post

print("=" * 60)
print("AUDIO TABLES WITH post.disp()")
print("=" * 60)
print()

# Create shorter excerpts for comparison (3 seconds)
short_excerpt = speech.excerpt(audio_path="tests/audio/spk/f10_script4_produced.wav", duration=3.0)

# Create a dictionary of audio comparisons
audio_dict = {
    "Original": short_excerpt,
    "Low-passed": short_excerpt.clone().low_pass(4000),
    "Pitch Shifted": short_excerpt.clone().pitch_shift(n_semitones=4),
}

print("Creating audio table with 3 versions:")
print(f"  Original: {audio_dict['Original'].audio_data.shape}")
print(f"  Low-passed: {audio_dict['Low-passed'].audio_data.shape}")
print(f"  Pitch Shifted: {audio_dict['Pitch Shifted'].audio_data.shape}")
print()

# Display the audio table (this will show as HTML in notebook)
post.disp(audio_dict)

print()
print("post.disp() creates interactive HTML audio players for comparison!")

## 10. Utility Functions

Test various utility functions from audiotools.core.util.

In [ ]:
print("=" * 60)
print("UTILITY FUNCTIONS")
print("=" * 60)
print()

# 1. find_audio - Find audio files in directories
print("-" * 60)
print("find_audio()")
print("-" * 60)
audio_files = util.find_audio("tests/audio/spk", ext=["wav"])
print(f"Found {len(audio_files)} WAV files in tests/audio/spk/")
for f in audio_files[:3]:  # Show first 3
    print(f"  {f}")
print()

# Also works with globs
audio_files_glob = util.find_audio("tests/audio/**/*.wav")
print(f"Using glob pattern 'tests/audio/**/*.wav': found {len(audio_files_glob)} files")
print()

# 2. hz_to_bin - Convert frequency to FFT bin
print("-" * 60)
print("hz_to_bin()")
print("-" * 60)
frequencies = torch.tensor([100, 1000, 5000, 10000])
n_fft = 2048
sample_rate = 44100

bins = util.hz_to_bin(frequencies, n_fft, sample_rate)
print(f"Input:")
print(f"  Frequencies: {frequencies.tolist()} Hz")
print(f"  n_fft: {n_fft}")
print(f"  Sample rate: {sample_rate} Hz")
print()
print(f"Output:")
print(f"  FFT bins: {bins.tolist()}")
print(f"  Bin -> Hz conversion check:")
for hz, bin_val in zip(frequencies.tolist(), bins.tolist()):
    reconstructed_hz = (bin_val / n_fft) * sample_rate
    print(f"    {hz} Hz -> bin {bin_val} -> {reconstructed_hz:.1f} Hz")
print()

# 3. sample_from_dist - Sample from distributions
print("-" * 60)
print("sample_from_dist()")
print("-" * 60)

# Uniform distribution
val = util.sample_from_dist(("uniform", 0.0, 1.0), state=42)
print(f"Uniform(0.0, 1.0): {val:.4f}")

# Constant value
val = util.sample_from_dist(("const", 5.0))
print(f"Const(5.0): {val}")

# Choice from list
val = util.sample_from_dist(("choice", [8, 16, 32, 64]), state=42)
print(f"Choice([8, 16, 32, 64]): {val}")
print()

# 4. seed - Set random seeds for reproducibility
print("-" * 60)
print("seed() - Reproducible randomness")
print("-" * 60)
util.seed(0)
random_1 = torch.randn(3)
print(f"With seed(0): {random_1.tolist()}")

util.seed(0)  # Same seed
random_2 = torch.randn(3)
print(f"With seed(0) again: {random_2.tolist()}")
print(f"Are they equal? {torch.allclose(random_1, random_2)}")

## 11. Metrics

Test audio quality metrics.

In [ ]:
# Create a degraded version
speech_degraded = speech_excerpt.clone()
speech_degraded = speech_degraded.low_pass(3000)
# speech_degraded = speech_degraded.apply_codec("MP3", bits_per_sample="64k")

# Resample both to 16kHz for metrics
speech_ref = speech_excerpt.clone().resample(16000)
speech_deg = speech_degraded.resample(16000)
speech_ref = speech_ref[...,:speech_deg.shape[-1]]
speech_deg = speech_deg[...,:speech_ref.shape[-1]]
print(f"speech_ref duration: {speech_ref.shape}s, speech_deg duration: {speech_deg.shape}s")
print("Computing audio quality metrics...")
print("(This may take a moment)\n")

In [ ]:
# STOI (Short-Time Objective Intelligibility)
stoi_score = metrics.quality.stoi(speech_ref, speech_deg)
print(f"STOI: {stoi_score}")


In [ ]:
# PESQ (Perceptual Evaluation of Speech Quality)
pesq_score = metrics.quality.pesq(speech_ref, speech_deg)
print(f"PESQ: {pesq_score}")

In [ ]:
# # ViSQOL (Virtual Speech Quality Objective Listener)
# visqol_score = metrics.quality.visqol(speech_ref, speech_deg)
# print(f"ViSQOL: {visqol_score}")

In [ ]:
# Distance metrics
l1_loss = metrics.distance.L1Loss()(speech_ref, speech_deg)
print(f"L1 Loss: {l1_loss.item():.6f}")

sisdr_loss = metrics.distance.SISDRLoss()(speech_ref, speech_deg)
print(f"SI-SDR Loss: {sisdr_loss.item():.6f}")

## 12. Batch Processing

Test batching multiple signals together.

In [ ]:
# Create multiple excerpts
excerpts = [speech.excerpt(audio_path="tests/audio/spk/f10_script4_produced.wav",duration=2.0) for _ in range(5)]
print(f"Created {len(excerpts)} excerpts")

# Batch signals
batched = AudioSignal.batch(excerpts)
print(f"Batched shape: {batched.audio_data.shape}")
print(f"Batch size: {batched.batch_size}")

In [ ]:
batch_transform = tfm.VolumeChange(db=("uniform", -6, 6))

transformed_list = []
for sig in batched:
    kwargs = batch_transform.instantiate(state=42, signal=sig)
    transformed_list.append(batch_transform(sig.clone(), **kwargs))

batched_transformed = AudioSignal.batch(transformed_list)
print(f"Batched transformed shape: {batched_transformed.audio_data.shape}")


## 13. Save/Export

Test saving audio to different formats.

In [ ]:
# Save to WAV
speech_excerpt.write("/tmp/test_output.wav")
print("Saved to /tmp/test_output.wav")

# Save to MP3
speech_excerpt.write("/tmp/test_output.mp3")
print("Saved to /tmp/test_output.mp3")

# Verify files were created
import os
print(f"\nWAV exists: {os.path.exists('/tmp/test_output.wav')}")
print(f"MP3 exists: {os.path.exists('/tmp/test_output.mp3')}")

## 14. Advanced: Custom Processing Pipeline

Demonstrate a complete audio processing pipeline.

In [ ]:
def audio_augmentation_pipeline(signal, seed=None):
    """Complete audio augmentation pipeline."""
    if seed is not None:
        util.seed(seed)
    
    # Clone input
    output = signal.clone()
    
    # 1. Normalize loudness
    output = output.normalize(db=-20)
    
    # 2. Add room acoustics
    rir_transform = tfm.RoomImpulseResponse(sources=["tests/audio/irs.csv"])
    kwargs = rir_transform.instantiate(state=seed, signal=output)
    output = rir_transform(output, **kwargs)
    
    # 3. Add background noise
    bg_transform = tfm.BackgroundNoise(sources=["tests/audio/noises.csv"], snr=("uniform", 10, 20))
    kwargs = bg_transform.instantiate(state=seed, signal=output)
    output = bg_transform(output, **kwargs)
    
    # 4. Apply codec
    output = output.apply_codec("MP3", bits_per_sample="128k")
    
    # 5. Final normalize
    output = output.normalize(db=-20)
    
    return output

# Apply pipeline
speech_processed = audio_augmentation_pipeline(speech_excerpt, seed=42)

# Visualize results
fig, axes = plt.subplots(2, 1, figsize=(12, 6))
speech_excerpt.specshow(ax=axes[0])
axes[0].set_title("Original")
speech_processed.specshow(ax=axes[1])
axes[1].set_title("Processed (Normalized + Reverb + Noise + Codec)")
plt.tight_layout()
plt.show()

print(f"Original loudness: {speech_excerpt.loudness()} LUFS")
print(f"Processed loudness: {speech_processed.loudness()} LUFS")

## Summary

This notebook demonstrated:

1. **Loading**: Multiple ways to load audio from files and CSV sources
2. **Visualization**: Waveform and spectrogram plotting with advanced options
3. **Basic Operations**: Cloning, resampling, mono conversion, excerpting
4. **DSP**: Filtering, windowing, overlap-add
5. **Effects**: Volume, pitch shift, time stretch, mixing, reverb, codec simulation, distortion
6. **Spectral**: STFT, mel spectrograms, MFCC, phase manipulation, masking
7. **Loudness**: Measurement and normalization
8. **Transforms**: Data augmentation for training (Compose, Choose, filtering)
9. **Spectral Losses**: MultiScaleSTFTLoss, MelSpectrogramLoss, PhaseLoss
10. **Audio Tables**: post.disp() for interactive HTML comparison
11. **Utility Functions**: find_audio, hz_to_bin, sample_from_dist, seed
12. **Metrics**: Quality assessment (STOI, PESQ, SI-SDR, L1 Loss)
13. **Batching**: Processing multiple signals together
14. **Export**: Saving to various formats
15. **Dataset Classes**: AudioLoader and AudioDataset for PyTorch training
    - AudioLoader: Load audio endlessly from sources
    - AudioDataset: PyTorch-compatible dataset
    - DataLoader integration: Batched training
    - Multiple loaders: Complex dataset creation
16. **Additional Signal Operations**: Advanced methods
    - zero_pad_to, truncate_samples: Padding and truncation
    - convolve: Direct convolution operations
    - equalizer: Mel-spaced equalization
    - hash, deepcopy, detach: Signal management
    - numpy, float, cpu, to: Device and type conversions

All functions were tested with real audio files from the tests directory!

## 15. Dataset Classes

AudioTools provides dataset classes for training machine learning models with audio data.

### AudioLoader - Load audio endlessly from sources

AudioLoader loads audio files from directories or CSV files for training.

In [ ]:
from audiotools.data.datasets import AudioLoader, AudioDataset

print("=" * 60)
print("AUDIOLOADER - Load audio from sources")
print("=" * 60)

# Create an AudioLoader pointing to test audio files
loader = AudioLoader(
    sources=["tests/audio/spk"],
    ext=["wav"],
    shuffle=True,
    shuffle_state=42
)

print(f"\nAudioLoader created:")
print(f"  Sources: {loader.sources}")
print(f"  Total audio files found: {len(loader.audio_indices)}")
print(f"  Audio lists: {len(loader.audio_lists)} source(s)")
print()

# Sample audio from the loader
state = util.random_state(42)
sample_rate = 16000
duration = 2.0

print(f"Loading sample with:")
print(f"  Sample rate: {sample_rate} Hz")
print(f"  Duration: {duration} seconds")
print()

# Load a sample
item = loader(
    state=state,
    sample_rate=sample_rate,
    duration=duration,
    loudness_cutoff=-40,
    num_channels=1
)

print(f"Loaded item:")
print(f"  Signal shape: {item['signal'].audio_data.shape}")
print(f"  Sample rate: {item['signal'].sample_rate} Hz")
print(f"  Duration: {item['signal'].duration:.3f}s")
print(f"  Source: {item['source']}")
print(f"  Path: {Path(item['path']).name}")
print(f"  Source index: {item['source_idx']}")
print(f"  Item index: {item['item_idx']}")
print()

# Load multiple samples
print("Loading 3 samples:")
for i in range(3):
    item = loader(state=state, sample_rate=16000, duration=1.0)
    print(f"  Sample {i+1}: {Path(item['path']).name} - {item['signal'].audio_data.shape}")

### AudioLoader with Transforms

AudioLoader can apply transforms to loaded audio.

In [ ]:
print("=" * 60)
print("AUDIOLOADER WITH TRANSFORMS")
print("=" * 60)

# Create loader with transform
loader_with_transform = AudioLoader(
    sources=["tests/audio/spk"],
    ext=["wav"],
    transform=tfm.Compose([
        tfm.VolumeChange(db=("uniform", -6, 6)),
        tfm.LowPass(cutoff=("choice", [4000, 8000, 16000])),
    ])
)

print(f"AudioLoader with transforms:")
print(f"  Transform: VolumeChange + LowPass")
print()

# Load a sample
state = util.random_state(42)
item = loader_with_transform(
    state=state,
    sample_rate=16000,
    duration=2.0,
)

print(f"Loaded item:")
print(f"  Signal shape: {item['signal'].audio_data.shape}")
print(f"  Has transform_args: {'transform_args' in item}")

if 'transform_args' in item:
    print(f"\n  Transform arguments:")
    for key, value in item['transform_args'].items():
        print(f"    {key}:")
        if isinstance(value, dict):
            for k, v in value.items():
                if isinstance(v, torch.Tensor):
                    print(f"      {k}: {v.item() if v.numel() == 1 else v.shape}")
                else:
                    print(f"      {k}: {v}")
        else:
            print(f"      {value}")
    
    # Apply the transform
    print(f"\n  Applying transform...")
    transformed_signal = loader_with_transform.transform(
        item['signal'].clone(),
        **item['transform_args']
    )
    print(f"  Transformed signal shape: {transformed_signal.audio_data.shape}")

### AudioDataset - Complete dataset for training

AudioDataset combines multiple AudioLoaders into a PyTorch-compatible dataset.

In [ ]:
print("=" * 60)
print("AUDIODATASET - PyTorch Dataset for Audio")
print("=" * 60)

# Create a simple dataset with a single loader
dataset = AudioDataset(
    loaders=AudioLoader(
        sources=["tests/audio/spk"],
        ext=["wav"],
        transform=tfm.LowPass(cutoff=("choice", [4000, 8000])),
    ),
    sample_rate=16000,
    n_examples=10,  # Small dataset for demo
    duration=2.0,
    num_channels=1,
    transform=tfm.VolumeNorm(("const", -20)),
)

print(f"AudioDataset created:")
print(f"  Length: {len(dataset)}")
print(f"  Sample rate: {dataset.sample_rate} Hz")
print(f"  Duration: {dataset.duration} seconds")
print(f"  Num channels: {dataset.num_channels}")
print()

# Get an item from the dataset
print("Getting item from dataset...")
item = dataset[0]

print(f"\nDataset item structure:")
print(f"  Keys: {list(item.keys())}")
print(f"  Signal shape: {item['signal'].audio_data.shape}")
print(f"  Signal sample rate: {item['signal'].sample_rate} Hz")
print(f"  Signal duration: {item['signal'].duration:.3f}s")
print(f"  Source: {item['source']}")
print(f"  Path: {Path(item['path']).name}")
print()

# Access multiple items
print("Accessing 3 items from dataset:")
for i in range(3):
    item = dataset[i]
    print(f"  Item {i}: shape={item['signal'].audio_data.shape}, path={Path(item['path']).name}")

### AudioDataset with PyTorch DataLoader

Use AudioDataset with PyTorch DataLoader for batched training.

In [ ]:
from torch.utils.data import DataLoader

print("=" * 60)
print("AUDIODATASET WITH PYTORCH DATALOADER")
print("=" * 60)

# Create dataset
train_dataset = AudioDataset(
    loaders=AudioLoader(
        sources=["tests/audio/spk"],
        ext=["wav"],
    ),
    sample_rate=16000,
    n_examples=20,
    duration=1.0,
    num_channels=1,
)

print(f"Dataset created:")
print(f"  Length: {len(train_dataset)}")
print()

# Create DataLoader with AudioDataset's collate function
dataloader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0,  # Set to 0 for notebook compatibility
    collate_fn=train_dataset.collate,
)

print(f"DataLoader created:")
print(f"  Batch size: 4")
print(f"  Number of batches: {len(dataloader)}")
print()

# Get a batch
batch = next(iter(dataloader))

print(f"Batch structure:")
print(f"  Keys: {list(batch.keys())}")
print(f"  Signal shape: {batch['signal'].audio_data.shape} (batch, channels, samples)")
print(f"  Signal is batched AudioSignal: {isinstance(batch['signal'], AudioSignal)}")
print(f"  Batch size: {batch['signal'].batch_size}")
print()

print(f"Individual items in batch:")
for i in range(batch['signal'].batch_size):
    item_signal = batch['signal'][i]
    print(f"  Item {i}: shape={item_signal.audio_data.shape}, path={Path(batch['path'][i]).name}")
print()

print("✓ Dataset is ready for training!")

### Multiple Loaders for Different Audio Sources

AudioDataset can use multiple loaders for different audio sources (e.g., speech + noise).

In [ ]:
print("=" * 60)
print("AUDIODATASET WITH MULTIPLE LOADERS")
print("=" * 60)

# Create dataset with multiple loaders (speech and noise)
multi_loader_dataset = AudioDataset(
    loaders={
        "speech": AudioLoader(
            sources=["tests/audio/spk"],
            ext=["wav"],
        ),
        "noise": AudioLoader(
            sources=["tests/audio/nz"],
            ext=["wav"],
        ),
    },
    sample_rate=16000,
    n_examples=10,
    duration=2.0,
    num_channels=1,
)

print(f"Multi-loader dataset created:")
print(f"  Loaders: {list(multi_loader_dataset.loaders.keys())}")
print(f"  Length: {len(multi_loader_dataset)}")
print()

# Get an item
item = multi_loader_dataset[0]

print(f"Item structure:")
print(f"  Keys: {list(item.keys())}")
print()

print(f"Speech loader output:")
print(f"  Signal shape: {item['speech']['signal'].audio_data.shape}")
print(f"  Path: {Path(item['speech']['path']).name}")
print()

print(f"Noise loader output:")
print(f"  Signal shape: {item['noise']['signal'].audio_data.shape}")
print(f"  Path: {Path(item['noise']['path']).name}")
print()

# Create a mix
speech_signal = item['speech']['signal']
noise_signal = item['noise']['signal']
mix = speech_signal.clone().mix(noise_signal, snr=10)

print(f"Created mixture:")
print(f"  Mix shape: {mix.audio_data.shape}")
print(f"  Mix duration: {mix.duration:.3f}s")
print(f"  SNR: 10 dB")
print()

print("✓ Multiple loaders allow complex dataset creation!")

## 16. Additional Signal Operations

Test additional AudioSignal methods not covered in previous sections.

## 16. Additional Signal Operations

Test additional AudioSignal methods not covered in previous sections.

### Padding and Truncation Methods

Advanced padding and truncation operations.

In [ ]:
print("=" * 60)
print("ZERO_PAD_TO - Pad to specific length")
print("=" * 60)

test_signal = speech_excerpt.clone()
original_length = test_signal.signal_length

print(f"Original:")
print(f"  Signal length: {original_length} samples")
print(f"  Shape: {test_signal.audio_data.shape}")
print()

# Pad to a specific length (after)
target_length = original_length + 5000
padded_after = test_signal.clone().zero_pad_to(target_length, mode="after")

print(f"After zero_pad_to({target_length}, mode='after'):")
print(f"  Signal length: {padded_after.signal_length} samples")
print(f"  Shape: {padded_after.audio_data.shape}")
print(f"  Added samples: {padded_after.signal_length - original_length}")
print()

# Pad to a specific length (before)
padded_before = test_signal.clone().zero_pad_to(target_length, mode="before")

print(f"After zero_pad_to({target_length}, mode='before'):")
print(f"  Signal length: {padded_before.signal_length} samples")
print(f"  Shape: {padded_before.audio_data.shape}")
print(f"  Added samples: {padded_before.signal_length - original_length}")
print()

# Truncate to specific number of samples
print("-" * 60)
print("TRUNCATE_SAMPLES - Truncate to specific sample count")
print("-" * 60)

truncate_length = original_length // 2
truncated = test_signal.clone().truncate_samples(truncate_length)

print(f"After truncate_samples({truncate_length}):")
print(f"  Original length: {original_length} samples")
print(f"  Truncated length: {truncated.signal_length} samples")
print(f"  Shape: {truncated.audio_data.shape}")
print(f"  Removed samples: {original_length - truncated.signal_length}")

### Convolution Operations

Direct convolution without using impulse response transforms.

In [ ]:
print("=" * 60)
print("CONVOLVE - Direct convolution with another signal")
print("=" * 60)

# Create signals for convolution
signal_to_convolve = speech_excerpt.clone()
ir_signal = ir.clone()

print(f"Signal 1 (speech):")
print(f"  Shape: {signal_to_convolve.audio_data.shape}")
print(f"  Duration: {signal_to_convolve.duration:.3f}s")
print()

print(f"Signal 2 (impulse response):")
print(f"  Shape: {ir_signal.audio_data.shape}")
print(f"  Duration: {ir_signal.duration:.3f}s")
print()

# Convolve the signals
convolved = signal_to_convolve.convolve(ir_signal.clone(), start_at_max=True)

print(f"After convolve():")
print(f"  Output shape: {convolved.audio_data.shape}")
print(f"  Output duration: {convolved.duration:.3f}s")
print(f"  start_at_max=True: Aligns IR to max to avoid delay")
print()

# Visualize
fig, axes = plt.subplots(3, 1, figsize=(12, 8))
signal_to_convolve.specshow(ax=axes[0])
axes[0].set_title("Original Signal")
ir_signal.specshow(ax=axes[1])
axes[1].set_title("Impulse Response")
convolved.specshow(ax=axes[2])
axes[2].set_title("Convolved Signal")
plt.tight_layout()
plt.show()

### Equalizer

Apply mel-spaced equalization curves to audio.

In [ ]:
print("=" * 60)
print("EQUALIZER - Mel-spaced EQ")
print("=" * 60)

test_signal = speech_excerpt.clone()

print(f"Input signal:")
print(f"  Shape: {test_signal.audio_data.shape}")
print(f"  Duration: {test_signal.duration:.3f}s")
print()

# Create EQ curves
n_bands = 8

# Boost high frequencies
eq_high_boost = torch.zeros(n_bands)
eq_high_boost[-3:] = 6  # Boost last 3 bands by 6dB

print(f"EQ curve (high boost): {eq_high_boost.tolist()}")
equalized_high = test_signal.clone().equalizer(eq_high_boost)

print(f"After equalizer (high boost):")
print(f"  Shape: {equalized_high.audio_data.shape}")
print(f"  Duration: {equalized_high.duration:.3f}s")
print()

# Boost low frequencies
eq_low_boost = torch.zeros(n_bands)
eq_low_boost[:3] = 6  # Boost first 3 bands by 6dB

print(f"EQ curve (low boost): {eq_low_boost.tolist()}")
equalized_low = test_signal.clone().equalizer(eq_low_boost)

print(f"After equalizer (low boost):")
print(f"  Shape: {equalized_low.audio_data.shape}")
print(f"  Duration: {equalized_low.duration:.3f}s")
print()

# V-shaped EQ
eq_v_shape = torch.tensor([4, 2, 0, -2, -2, 0, 2, 4], dtype=torch.float32)

print(f"EQ curve (V-shaped): {eq_v_shape.tolist()}")
equalized_v = test_signal.clone().equalizer(eq_v_shape)

print(f"After equalizer (V-shaped):")
print(f"  Shape: {equalized_v.audio_data.shape}")
print(f"  Duration: {equalized_v.duration:.3f}s")
print()

# Visualize EQ effects
fig, axes = plt.subplots(4, 1, figsize=(12, 10))
test_signal.specshow(ax=axes[0])
axes[0].set_title("Original")
equalized_high.specshow(ax=axes[1])
axes[1].set_title(f"High Boost: {eq_high_boost.tolist()}")
equalized_low.specshow(ax=axes[2])
axes[2].set_title(f"Low Boost: {eq_low_boost.tolist()}")
equalized_v.specshow(ax=axes[3])
axes[3].set_title(f"V-Shaped: {eq_v_shape.tolist()}")
plt.tight_layout()
plt.show()

### Hash, Deepcopy, and Tensor Operations

Utility methods for signal manipulation and management.

In [ ]:
print("=" * 60)
print("HASH - Compute hash of audio signal")
print("=" * 60)

signal1 = speech_excerpt.clone()
signal2 = speech_excerpt.clone()
signal3 = speech_excerpt.clone().volume_change(db=6)

hash1 = signal1.hash()
hash2 = signal2.hash()
hash3 = signal3.hash()

print(f"Signal 1 hash: {hash1}")
print(f"Signal 2 hash (same audio): {hash2}")
print(f"Signal 3 hash (modified): {hash3}")
print()
print(f"Hash 1 == Hash 2: {hash1 == hash2} (same audio)")
print(f"Hash 1 == Hash 3: {hash1 == hash3} (different audio)")
print()

# Deepcopy vs Clone
print("-" * 60)
print("DEEPCOPY vs CLONE")
print("-" * 60)

original = speech_excerpt.clone()

# Clone
cloned = original.clone()
print(f"Clone:")
print(f"  Creates a copy with same audio_data")
print(f"  Original shape: {original.audio_data.shape}")
print(f"  Cloned shape: {cloned.audio_data.shape}")
print(f"  Same object: {cloned is original}")
print()

# Deepcopy
deepcopied = original.deepcopy()
print(f"Deepcopy:")
print(f"  Creates a complete deep copy")
print(f"  Original shape: {original.audio_data.shape}")
print(f"  Deepcopied shape: {deepcopied.audio_data.shape}")
print(f"  Same object: {deepcopied is original}")
print()

# Detach from computation graph
print("-" * 60)
print("DETACH - Detach from computation graph")
print("-" * 60)

signal_with_grad = speech_excerpt.clone()
signal_with_grad.audio_data.requires_grad = True

print(f"Before detach():")
print(f"  requires_grad: {signal_with_grad.audio_data.requires_grad}")
print()

detached_signal = signal_with_grad.detach()

print(f"After detach():")
print(f"  requires_grad: {detached_signal.audio_data.requires_grad}")
print(f"  Shape preserved: {detached_signal.audio_data.shape}")
print(f"  Useful for inference without gradients")

### Device and Type Conversions

Move signals between devices and convert data types.

In [ ]:
print("=" * 60)
print("DEVICE AND TYPE CONVERSIONS")
print("=" * 60)

test_signal = speech_excerpt.clone()

print(f"Original signal:")
print(f"  Device: {test_signal.device}")
print(f"  Dtype: {test_signal.audio_data.dtype}")
print(f"  Shape: {test_signal.audio_data.shape}")
print()

# Convert to float32 explicitly
print("-" * 60)
print("FLOAT() - Ensure float32 dtype")
print("-" * 60)

float_signal = test_signal.clone().float()
print(f"After .float():")
print(f"  Dtype: {float_signal.audio_data.dtype}")
print(f"  Shape: {float_signal.audio_data.shape}")
print()

# Move to CPU
print("-" * 60)
print("CPU() - Move to CPU device")
print("-" * 60)

cpu_signal = test_signal.clone().cpu()
print(f"After .cpu():")
print(f"  Device: {cpu_signal.device}")
print(f"  Shape: {cpu_signal.audio_data.shape}")
print()

# Convert to numpy
print("-" * 60)
print("NUMPY() - Convert to numpy array")
print("-" * 60)

numpy_data = test_signal.numpy()
print(f"After .numpy():")
print(f"  Type: {type(numpy_data)}")
print(f"  Shape: {numpy_data.shape}")
print(f"  Dtype: {numpy_data.dtype}")
print(f"  Min: {numpy_data.min():.6f}")
print(f"  Max: {numpy_data.max():.6f}")
print()

# Move to device using .to()
print("-" * 60)
print("TO(device) - Move to specific device")
print("-" * 60)

# Move to CPU using .to()
cpu_signal_to = test_signal.clone().to("cpu")
print(f"After .to('cpu'):")
print(f"  Device: {cpu_signal_to.device}")
print(f"  Shape: {cpu_signal_to.audio_data.shape}")
print()

print("Note: .cuda() is also available for GPU operations")